- **g(n):** lo que llevamos pagado desde el inicio hasta n (costo real).
- **h(n):** lo que creemos que falta de n a la meta (heurística, es una estimación).

Esta función la usamos en varios algoritmos para armar el camino al final. Cada vez que descubrimos un nodo guardamos de quién vino (su padre), y al llegar a la meta nos regresamos de padre en padre hasta el inicio.

In [1]:
def reconstruir_camino(padres, meta):
    camino = [meta]
    while padres[camino[-1]] is not None:
        camino.append(padres[camino[-1]])
    camino.reverse()
    return camino

## 1. Búsqueda en anchura (BFS)

**Qué usa:** una cola (FIFO, el primero que entra es el primero que sale) y un conjunto de visitados.

**Pesos:** no los toma en cuenta. Para BFS todas las aristas valen lo mismo, solo cuenta el número de pasos.

**Qué compara:** lo único que revisa es si el vecino ya está en visitados. No compara costos ni distancias.

**Paso a paso:**

1. Metemos el nodo inicial a la cola y lo marcamos como visitado.
2. Sacamos el nodo que está al frente de la cola.
3. Revisamos sus vecinos uno por uno. Si el vecino no está en visitados, lo marcamos y lo formamos al final de la cola. Si ya estaba, lo ignoramos.
4. Repetimos desde el paso 2 hasta que la cola quede vacía (o hasta sacar la meta).

**Cómo sale el siguiente nodo:** siempre es el que está al frente de la cola. Como los vecinos nuevos se forman al final, primero terminamos todos los nodos a distancia 1, luego los de distancia 2, y así. Por eso se dice que va "por niveles", como las ondas en el agua.

In [2]:
from collections import deque

def bfs(grafo, inicio):
    visitados = {inicio}
    cola = deque([inicio])
    orden = []

    while cola:
        nodo = cola.popleft()          # sacamos el del frente
        orden.append(nodo)
        print(f"Sacamos {nodo}, sus vecinos son {grafo[nodo]}")

        for vecino in grafo[nodo]:
            if vecino not in visitados:
                visitados.add(vecino)
                cola.append(vecino)    # lo formamos al final
                print(f"   {vecino} no estaba visitado -> lo metemos a la cola")
            else:
                print(f"   {vecino} ya estaba visitado -> lo ignoramos")

        print(f"   cola ahora: {list(cola)}\n")

    return orden


grafo = {
    0: [1, 2],
    1: [2],
    2: [0, 3],
    3: [3],
}

print("Orden BFS:", bfs(grafo, 2))

Sacamos 2, sus vecinos son [0, 3]
   0 no estaba visitado -> lo metemos a la cola
   3 no estaba visitado -> lo metemos a la cola
   cola ahora: [0, 3]

Sacamos 0, sus vecinos son [1, 2]
   1 no estaba visitado -> lo metemos a la cola
   2 ya estaba visitado -> lo ignoramos
   cola ahora: [3, 1]

Sacamos 3, sus vecinos son [3]
   3 ya estaba visitado -> lo ignoramos
   cola ahora: [1]

Sacamos 1, sus vecinos son [2]
   2 ya estaba visitado -> lo ignoramos
   cola ahora: []

Orden BFS: [2, 0, 3, 1]


### BFS para encontrar el camino más corto (en número de pasos)

Es lo mismo, pero ahora guardamos el padre de cada nodo y paramos cuando sacamos la meta. Como BFS va por niveles, la primera vez que llegamos a la meta es por el camino con menos aristas.

In [3]:
def bfs_camino(grafo, inicio, meta):
    padres = {inicio: None}      # también nos sirve como visitados
    cola = deque([inicio])

    while cola:
        nodo = cola.popleft()
        if nodo == meta:
            return reconstruir_camino(padres, meta)

        for vecino in grafo[nodo]:
            if vecino not in padres:
                padres[vecino] = nodo
                cola.append(vecino)

    return None   # no hay camino


print("Camino de 2 a 1:", bfs_camino(grafo, 2, 1))

Camino de 2 a 1: [2, 0, 1]


## 2. Búsqueda en profundidad (DFS)

**Qué usa:** una pila (LIFO, el último que entra es el primero que sale). En la versión recursiva la pila es la propia pila de llamadas de Python. También usa visitados.

**Pesos:** tampoco los toma en cuenta.

**Qué compara:** igual que BFS, solo revisa si el vecino ya fue visitado.

**Paso a paso:**

1. Visitamos el nodo actual y lo marcamos.
2. Tomamos su primer vecino no visitado y nos metemos en él (llamada recursiva). Ahí volvemos a hacer lo mismo: bajamos al primer vecino no visitado, y así, alejándonos lo más posible del inicio.
3. Cuando llegamos a un nodo que ya no tiene vecinos sin visitar (callejón sin salida), la función termina y regresamos al nodo anterior (backtracking).
4. En el nodo anterior seguimos con el siguiente vecino que nos faltaba.

**Cómo sale el siguiente nodo:** es el vecino no visitado más reciente. No terminamos un nivel antes de pasar al siguiente, sino que seguimos una rama hasta el fondo.


In [4]:
def dfs(grafo, nodo, visitados=None, orden=None, nivel=0):
    if visitados is None:
        visitados = set()
        orden = []

    sangria = "   " * nivel
    visitados.add(nodo)
    orden.append(nodo)
    print(f"{sangria}Visitamos {nodo}")

    for vecino in grafo[nodo]:
        if vecino not in visitados:
            print(f"{sangria}   bajamos de {nodo} a {vecino}")
            dfs(grafo, vecino, visitados, orden, nivel + 1)
        else:
            print(f"{sangria}   {vecino} ya visitado, no bajamos")

    print(f"{sangria}{nodo} ya no tiene vecinos nuevos -> regresamos")
    return orden


print("\nOrden DFS:", dfs(grafo, 2))

Visitamos 2
   bajamos de 2 a 0
   Visitamos 0
      bajamos de 0 a 1
      Visitamos 1
         2 ya visitado, no bajamos
      1 ya no tiene vecinos nuevos -> regresamos
      2 ya visitado, no bajamos
   0 ya no tiene vecinos nuevos -> regresamos
   bajamos de 2 a 3
   Visitamos 3
      3 ya visitado, no bajamos
   3 ya no tiene vecinos nuevos -> regresamos
2 ya no tiene vecinos nuevos -> regresamos

Orden DFS: [2, 0, 1, 3]


### DFS con pila explícita

Es la misma idea sin recursión. Metemos los vecinos a la pila en orden inverso para que el primer vecino quede hasta arriba y sea el que sale primero (así el orden queda igual que en la versión recursiva).

Aquí sí marcamos como visitado al *sacar* el nodo, porque un nodo puede quedar en la pila más de una vez y solo lo procesamos la primera vez que sale.

In [5]:
def dfs_pila(grafo, inicio):
    visitados = set()
    pila = [inicio]
    orden = []

    while pila:
        nodo = pila.pop()              # sacamos el de arriba
        if nodo in visitados:
            continue
        visitados.add(nodo)
        orden.append(nodo)

        for vecino in reversed(grafo[nodo]):
            if vecino not in visitados:
                pila.append(vecino)
        print(f"Sacamos {nodo}, pila ahora: {pila}")

    return orden


print("Orden DFS (pila):", dfs_pila(grafo, 2))

Sacamos 2, pila ahora: [3, 0]
Sacamos 0, pila ahora: [3, 1]
Sacamos 1, pila ahora: [3]
Sacamos 3, pila ahora: []
Orden DFS (pila): [2, 0, 1, 3]


## 3. Costo uniforme (UCS)

Aquí ya entran los **pesos** de las aristas. Sirve cuando no todas las acciones cuestan lo mismo.

**Qué usa:** una cola de prioridad (`heapq`). Cada elemento es `(costo acumulado, nodo, camino)` y siempre sale el de menor costo acumulado g(n). También guardamos un diccionario con el mejor costo que conocemos para llegar a cada nodo.

**Qué compara:**

- Para escoger el siguiente nodo: compara el **g(n) de todos los nodos de la frontera** y saca el menor.
- Al revisar un vecino: calcula `nuevo costo = costo del padre + peso de la arista` y lo compara contra el **mejor costo que ya teníamos guardado para ese vecino**. Solo si el nuevo es menor lo guardamos y lo metemos a la frontera (esto es la "regla de oro" de la diapositiva, actualizar si encontramos un camino más barato).

**Paso a paso:**

1. Metemos el inicio con costo 0.
2. Sacamos de la frontera el nodo con menor costo acumulado.
3. Si ese nodo es la meta, terminamos. El costo con el que salió es el mínimo.
4. Si no, para cada vecino calculamos su nuevo costo y lo comparamos contra el que teníamos. Si mejora, lo guardamos y lo metemos a la frontera.
5. Regresamos al paso 2.

**Por qué revisamos la meta al sacarla y no al generarla:** en el ejemplo, D se genera primero con costo 6 (por A → C → D). Si paráramos ahí nos quedaríamos con un camino caro. Esperando a que D salga de la cola nos aseguramos de que ya no hay nada más barato pendiente.

**Nodos repetidos en la cola:** cuando mejoramos el costo de un nodo no borramos la entrada vieja, solo metemos otra. Cuando la vieja sale, vemos que su costo es peor que el guardado y la saltamos.


In [6]:
import heapq

def ucs(grafo, inicio, meta):
    frontera = [(0, inicio, [inicio])]
    mejor_costo = {inicio: 0}

    while frontera:
        print(f"Frontera: {[(c, n) for c, n, _ in sorted(frontera)]}")
        costo, nodo, camino = heapq.heappop(frontera)
        print(f"Sacamos {nodo} (el de menor costo acumulado = {costo}), camino {camino}")

        if nodo == meta:
            return camino, costo

        if costo > mejor_costo[nodo]:
            print(f"   esta entrada es vieja, ya conocemos {nodo} con costo {mejor_costo[nodo]} -> la saltamos\n")
            continue

        for vecino, peso in grafo[nodo].items():
            nuevo = costo + peso
            anterior = mejor_costo.get(vecino)
            if anterior is None or nuevo < anterior:
                texto = "no lo teníamos" if anterior is None else f"antes costaba {anterior}"
                print(f"   {vecino}: {costo} + {peso} = {nuevo} ({texto}) -> lo guardamos")
                mejor_costo[vecino] = nuevo
                heapq.heappush(frontera, (nuevo, vecino, camino + [vecino]))
            else:
                print(f"   {vecino}: {costo} + {peso} = {nuevo}, pero ya lo tenemos con {anterior} -> no mejora")
        print()

    return None, float("inf")


grafo_pesos = {
    "A": {"B": 5, "C": 1},
    "B": {"D": 1},
    "C": {"B": 2, "D": 5},
    "D": {},
}

camino, costo = ucs(grafo_pesos, "A", "D")
print("\nMejor camino:", " -> ".join(camino), "| costo:", costo)

Frontera: [(0, 'A')]
Sacamos A (el de menor costo acumulado = 0), camino ['A']
   B: 0 + 5 = 5 (no lo teníamos) -> lo guardamos
   C: 0 + 1 = 1 (no lo teníamos) -> lo guardamos

Frontera: [(1, 'C'), (5, 'B')]
Sacamos C (el de menor costo acumulado = 1), camino ['A', 'C']
   B: 1 + 2 = 3 (antes costaba 5) -> lo guardamos
   D: 1 + 5 = 6 (no lo teníamos) -> lo guardamos

Frontera: [(3, 'B'), (5, 'B'), (6, 'D')]
Sacamos B (el de menor costo acumulado = 3), camino ['A', 'C', 'B']
   D: 3 + 1 = 4 (antes costaba 6) -> lo guardamos

Frontera: [(4, 'D'), (5, 'B'), (6, 'D')]
Sacamos D (el de menor costo acumulado = 4), camino ['A', 'C', 'B', 'D']

Mejor camino: A -> C -> B -> D | costo: 4


Si comparamos con BFS en el mismo grafo, BFS escogería A → C → D porque son solo 2 pasos, pero cuesta 6. UCS no cuenta pasos, cuenta costo.

UCS es prácticamente el algoritmo de Dijkstra, solo que se detiene al llegar a la meta en vez de calcular la distancia a todos los nodos.

In [7]:
grafo_sin_pesos = {n: list(v.keys()) for n, v in grafo_pesos.items()}
print("BFS diría:", bfs_camino(grafo_sin_pesos, "A", "D"))

BFS diría: ['A', 'B', 'D']


## 4. Profundidad limitada (DLS)

Es un DFS con un **límite L** de profundidad. Sirve para que DFS no se pierda en ramas infinitas o en ciclos.

**Qué usa:** recursión (pila) y un contador de cuánto límite nos queda.

**Qué compara:**

- Si el nodo actual es la meta.
- El límite que nos queda contra 0. Si llegamos a 0 ya no expandimos ese nodo aunque tenga hijos (lo tratamos como callejón sin salida).

**Paso a paso** (siguiendo el pseudocódigo de la diapositiva):

1. Si el nodo es la meta, regresamos éxito.
2. Si el límite es 0, regresamos **corte** (no sabemos si hay solución más abajo, simplemente no bajamos).
3. Si no, llamamos a DLS en cada hijo con límite - 1.
4. Si algún hijo encontró la meta, regresamos ese resultado.
5. Si ningún hijo la encontró pero alguno regresó corte, regresamos corte. Si todos fallaron sin corte, regresamos **fallo** (ahí sí sabemos que no hay solución por esa rama).

**Por qué distinguir corte y fallo:** corte significa "no la encontré, pero tal vez con un límite más grande sí". Fallo significa "aquí de plano no hay". Eso lo usa IDS para saber si vale la pena subir el límite.


In [8]:
CORTE = "corte"
FALLO = "fallo"

def dls(grafo, nodo, meta, limite, camino=None, verbose=True):
    if camino is None:
        camino = [nodo]
    sangria = "   " * (len(camino) - 1)

    if nodo == meta:
        if verbose:
            print(f"{sangria}{nodo} es la meta")
        return camino

    if limite == 0:
        if verbose:
            print(f"{sangria}{nodo}: límite en 0 -> corte (no expandimos sus hijos)")
        return CORTE

    if verbose:
        print(f"{sangria}Expandimos {nodo} (límite que queda: {limite})")

    hubo_corte = False
    for hijo in grafo[nodo]:
        resultado = dls(grafo, hijo, meta, limite - 1, camino + [hijo], verbose)
        if resultado == CORTE:
            hubo_corte = True
        elif resultado != FALLO:
            return resultado

    return CORTE if hubo_corte else FALLO


grafo_ciclo = {
    "A": ["B"],
    "B": ["C"],
    "C": ["A", "D"],
    "D": [],
}

print("Con L = 2")
print("Resultado:", dls(grafo_ciclo, "A", "D", 2))
print("\nCon L = 3")
print("Resultado:", dls(grafo_ciclo, "A", "D", 3))

Con L = 2
Expandimos A (límite que queda: 2)
   Expandimos B (límite que queda: 1)
      C: límite en 0 -> corte (no expandimos sus hijos)
Resultado: corte

Con L = 3
Expandimos A (límite que queda: 3)
   Expandimos B (límite que queda: 2)
      Expandimos C (límite que queda: 1)
         A: límite en 0 -> corte (no expandimos sus hijos)
         D es la meta
Resultado: ['A', 'B', 'C', 'D']


## 5. Profundidad iterativa (IDS)

Es correr DLS varias veces subiendo el límite: L = 0, luego 1, luego 2, etc., hasta encontrar la meta.

**Qué compara:** el resultado de cada DLS. Si fue corte, subimos el límite y volvemos a empezar desde el inicio. Si fue fallo, ya no tiene caso seguir porque no hay solución. Si encontró la meta, terminamos.

**Por qué conviene:** junta lo bueno de los dos. Usa poca memoria como DFS (O(bd)) y encuentra la meta más cercana como BFS, porque prueba primero todos los límites chicos. Parece que repite mucho trabajo, pero la mayoría de los nodos están en el último nivel, así que los niveles de arriba casi no pesan.

In [9]:
def ids(grafo, inicio, meta, limite_max=20):
    for limite in range(limite_max + 1):
        resultado = dls(grafo, inicio, meta, limite, verbose=False)
        print(f"L = {limite}: {resultado}")
        if resultado == FALLO:
            return None
        if resultado != CORTE:
            return resultado
    return None


print("Camino encontrado:", ids(grafo_ciclo, "A", "D"))

L = 0: corte
L = 1: corte
L = 2: corte
L = 3: ['A', 'B', 'C', 'D']
Camino encontrado: ['A', 'B', 'C', 'D']


## 6. Búsqueda bidireccional

Hacemos dos BFS al mismo tiempo: una hacia adelante desde el inicio y otra hacia atrás desde la meta. Paramos cuando se encuentran.

**Qué usa:** dos colas y dos diccionarios de padres (uno por lado). El grafo tiene que poder recorrerse al revés, aquí usamos uno no dirigido.

**Qué compara:** cada vez que un lado descubre un nodo nuevo, revisa si ese nodo **ya lo había descubierto el otro lado**. Si sí, ese es el punto de encuentro.

**Paso a paso:**

1. Metemos el inicio en la cola de adelante y la meta en la cola de atrás.
2. Expandimos un nivel completo del lado de adelante. Por cada vecino nuevo revisamos si está en los padres del lado de atrás.
3. Si no se encontraron, expandimos un nivel del lado de atrás y hacemos la misma revisión contra el lado de adelante.
4. Alternamos hasta que se encuentren.
5. Armamos el camino: del inicio al punto de encuentro con los padres de adelante, y del punto de encuentro a la meta con los padres de atrás.

**Por qué conviene:** cada búsqueda solo llega a la mitad de la profundidad, y b^(d/2) + b^(d/2) es mucho menor que b^d.

In [10]:
def expandir_nivel(grafo, cola, padres_este, padres_otro, lado):
    for _ in range(len(cola)):
        nodo = cola.popleft()
        for vecino in grafo[nodo]:
            if vecino not in padres_este:
                padres_este[vecino] = nodo
                cola.append(vecino)
                print(f"   [{lado}] desde {nodo} descubrimos {vecino}")
                if vecino in padres_otro:
                    print(f"   {vecino} ya lo había visto el otro lado -> se encontraron")
                    return vecino
    return None


def bidireccional(grafo, inicio, meta):
    if inicio == meta:
        return [inicio]

    padres_ini = {inicio: None}
    padres_meta = {meta: None}
    cola_ini = deque([inicio])
    cola_meta = deque([meta])

    while cola_ini and cola_meta:
        print("Expandimos un nivel desde el inicio")
        encuentro = expandir_nivel(grafo, cola_ini, padres_ini, padres_meta, "inicio")
        if encuentro is None:
            print("Expandimos un nivel desde la meta")
            encuentro = expandir_nivel(grafo, cola_meta, padres_meta, padres_ini, "meta")

        if encuentro is not None:
            mitad_1 = reconstruir_camino(padres_ini, encuentro)       # inicio ... encuentro
            mitad_2 = reconstruir_camino(padres_meta, encuentro)      # meta ... encuentro
            mitad_2.reverse()                                         # encuentro ... meta
            return mitad_1 + mitad_2[1:]

    return None


grafo_nd = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A", "E"],
    "D": ["B", "F"],
    "E": ["C", "F"],
    "F": ["D", "E", "G"],
    "G": ["F"],
}

print("\nCamino:", bidireccional(grafo_nd, "A", "G"))

Expandimos un nivel desde el inicio
   [inicio] desde A descubrimos B
   [inicio] desde A descubrimos C
Expandimos un nivel desde la meta
   [meta] desde G descubrimos F
Expandimos un nivel desde el inicio
   [inicio] desde B descubrimos D
   [inicio] desde C descubrimos E
Expandimos un nivel desde la meta
   [meta] desde F descubrimos D
   D ya lo había visto el otro lado -> se encontraron

Camino: ['A', 'B', 'D', 'F', 'G']


## Heurística h(n)

A partir de aquí los algoritmos son **informados**: además del grafo tienen una función h(n) que estima cuánto falta de n a la meta. Por ejemplo, la distancia en línea recta entre dos ciudades. Siempre h(meta) = 0.

- UCS mira hacia atrás: g(n), lo que ya pagamos.
- Greedy mira hacia adelante: h(n), lo que creemos que falta.
- A* mira las dos: f(n) = g(n) + h(n).

## 7. Primero el mejor (Greedy Best-First Search)

**Qué usa:** una cola de prioridad ordenada **solo por h(n)** y un conjunto de visitados.

**Pesos:** no los usa para decidir. Los sumamos aquí solo para ver cuánto costó el camino que escogió, pero no influyen en la elección.

**Qué compara:** la h de todos los nodos de la frontera. Saca el que parece estar más cerca de la meta, sin importar cuánto costó llegar a él.

**Paso a paso:**

1. Metemos el inicio con su h.
2. Sacamos el nodo con menor h.
3. Si es la meta, terminamos.
4. Si no, metemos a la frontera sus vecinos no visitados, cada uno con su h.
5. Regresamos al paso 2.

In [11]:
def greedy(grafo, h, inicio, meta):
    frontera = [(h[inicio], inicio, [inicio], 0)]
    visitados = set()

    while frontera:
        print(f"Frontera (h, nodo): {[(hh, n) for hh, n, _, _ in sorted(frontera)]}")
        _, nodo, camino, costo = heapq.heappop(frontera)
        print(f"Sacamos {nodo} porque tiene la h más chica ({h[nodo]})")

        if nodo == meta:
            return camino, costo
        visitados.add(nodo)

        for vecino, peso in grafo[nodo].items():
            if vecino not in visitados:
                print(f"   metemos {vecino} con h = {h[vecino]}")
                heapq.heappush(frontera, (h[vecino], vecino, camino + [vecino], costo + peso))
        print()

    return None, float("inf")


grafo_greedy = {
    "A": {"B": 5, "C": 50},
    "B": {"D": 12},
    "C": {"D": 3},
    "D": {},
}
h_greedy = {"A": 12, "B": 10, "C": 2, "D": 0}

camino, costo = greedy(grafo_greedy, h_greedy, "A", "D")
print("\nGreedy encontró:", " -> ".join(camino), "| costo real:", costo)

Frontera (h, nodo): [(12, 'A')]
Sacamos A porque tiene la h más chica (12)
   metemos B con h = 10
   metemos C con h = 2

Frontera (h, nodo): [(2, 'C'), (10, 'B')]
Sacamos C porque tiene la h más chica (2)
   metemos D con h = 0

Frontera (h, nodo): [(0, 'D'), (10, 'B')]
Sacamos D porque tiene la h más chica (0)

Greedy encontró: A -> C -> D | costo real: 53


Encontró una ruta, y rápido, pero no la más barata (A → B → D cuesta 17). Por eso Greedy **no es óptimo**. Se usa cuando nos importa encontrar cualquier ruta rápido y no la perfecta.

## 8. A* (A estrella)

**Qué usa:** una cola de prioridad ordenada por **f(n) = g(n) + h(n)** y un diccionario con el mejor g conocido para cada nodo.

**Qué compara:**

- Para escoger el siguiente nodo: la f de todos los nodos en la frontera, sale el de f más chica. O sea, balancea lo que ya pagamos (g) con lo que creemos que falta (h).
- Al revisar un vecino: calculamos `nuevo g = g del padre + peso de la arista` y lo comparamos contra el **g que ya teníamos para ese vecino**. Si es menor, lo actualizamos, calculamos su f = nuevo g + h(vecino) y lo metemos a la frontera.

Es exactamente el mismo esqueleto que UCS, lo único que cambia es la prioridad: UCS ordena por g y A* ordena por g + h.

**Paso a paso:**

1. Metemos el inicio con g = 0 y f = h(inicio).
2. Sacamos el nodo con menor f.
3. Si es la meta, terminamos.
4. Para cada vecino calculamos el nuevo g. Si es mejor que el guardado, lo guardamos y lo metemos con f = g + h.
5. Regresamos al paso 2.

**Condición para que sea óptimo:** h tiene que ser **admisible**, o sea, nunca debe sobreestimar lo que realmente falta. Si además es **consistente** (h(n) ≤ costo(n, n') + h(n')), no hace falta re-explorar nodos.

**Detalle:** si ponemos h(n) = 0 para todos, A* se vuelve UCS.


In [12]:
def a_estrella(grafo, h, inicio, meta, w=1, verbose=True):
    g = {inicio: 0}
    padres = {inicio: None}
    frontera = [(w * h[inicio], inicio)]
    expandidos = 0

    while frontera:
        if verbose:
            print(f"Frontera (f, nodo): {sorted(frontera)}")
        f, nodo = heapq.heappop(frontera)

        if f > g[nodo] + w * h[nodo]:
            continue   # entrada vieja, ya encontramos algo mejor para este nodo

        if verbose:
            print(f"Sacamos {nodo}: f = g + h = {g[nodo]} + {w * h[nodo]} = {f}")

        if nodo == meta:
            return reconstruir_camino(padres, meta), g[meta], expandidos

        expandidos += 1
        for vecino, peso in grafo[nodo].items():
            nuevo_g = g[nodo] + peso
            if vecino not in g or nuevo_g < g[vecino]:
                g[vecino] = nuevo_g
                padres[vecino] = nodo
                nuevo_f = nuevo_g + w * h[vecino]
                heapq.heappush(frontera, (nuevo_f, vecino))
                if verbose:
                    print(f"   {vecino}: g = {g[nodo]} + {peso} = {nuevo_g}, h = {w * h[vecino]}, f = {nuevo_f}")
            elif verbose:
                print(f"   {vecino}: g nuevo {nuevo_g} no mejora al que ya teníamos ({g[vecino]})")
        if verbose:
            print()

    return None, float("inf"), expandidos


grafo_a = {
    "A": {"B": 2, "C": 4},
    "B": {"D": 5},
    "C": {"D": 1},
    "D": {},
}
h_a = {"A": 6, "B": 4, "C": 1, "D": 0}

camino, costo, _ = a_estrella(grafo_a, h_a, "A", "D")
print("\nA* encontró:", " -> ".join(camino), "| costo:", costo)

Frontera (f, nodo): [(6, 'A')]
Sacamos A: f = g + h = 0 + 6 = 6
   B: g = 0 + 2 = 2, h = 4, f = 6
   C: g = 0 + 4 = 4, h = 1, f = 5

Frontera (f, nodo): [(5, 'C'), (6, 'B')]
Sacamos C: f = g + h = 4 + 1 = 5
   D: g = 4 + 1 = 5, h = 0, f = 5

Frontera (f, nodo): [(5, 'D'), (6, 'B')]
Sacamos D: f = g + h = 5 + 0 = 5

A* encontró: A -> C -> D | costo: 5


Ahora corremos A* en el mismo grafo donde Greedy se equivocó. Como A* también toma en cuenta g, al ver que C ya costó 50 prefiere irse por B.

In [13]:
camino, costo, _ = a_estrella(grafo_greedy, h_greedy, "A", "D")
print("\nA* encontró:", " -> ".join(camino), "| costo:", costo)

Frontera (f, nodo): [(12, 'A')]
Sacamos A: f = g + h = 0 + 12 = 12
   B: g = 0 + 5 = 5, h = 10, f = 15
   C: g = 0 + 50 = 50, h = 2, f = 52

Frontera (f, nodo): [(15, 'B'), (52, 'C')]
Sacamos B: f = g + h = 5 + 10 = 15
   D: g = 5 + 12 = 17, h = 0, f = 17

Frontera (f, nodo): [(17, 'D'), (52, 'C')]
Sacamos D: f = g + h = 17 + 0 = 17

A* encontró: A -> B -> D | costo: 17


## 9. A* pesado (Weighted A*)

Es A* pero multiplicando la heurística por un peso w > 1:

f(n) = g(n) + w · h(n)

**Qué cambia en las comparaciones:** la frontera se sigue ordenando por f, pero ahora h pesa más que g. Entre más grande es w, más se parece a Greedy: le hace más caso a "lo que falta" que a "lo que ya pagamos".

- w = 1: A* normal, óptimo pero explora más nodos.
- w > 1: explora menos nodos y llega más rápido, pero el camino puede salir un poco más largo.
- Se garantiza que el costo encontrado no pasa de w veces el óptimo. Con w = 1.5 el camino es, cuando mucho, 50% más largo.

Para ver la diferencia necesitamos algo más grande que 4 nodos. Usamos un laberinto en cuadrícula: `S` es el inicio, `G` la meta y `#` son paredes. Cada paso cuesta 1 y la heurística es la distancia Manhattan (|fila1 - fila2| + |col1 - col2|), que es admisible porque nunca podemos llegar en menos pasos que esos.

In [14]:
mapa = [
    "S.........#.........",
    ".########.#.######..",
    ".#......#.#......#..",
    ".#.####.#.######.#..",
    ".#.#..#.#......#.#..",
    ".#.#..#.######.#.#..",
    ".#.#..#........#.#..",
    ".#.####.########.#..",
    ".#...............#..",
    ".#################..",
    "...................G",
]


def mapa_a_grafo(mapa):
    grafo = {}
    inicio = meta = None
    filas, cols = len(mapa), len(mapa[0])
    for i in range(filas):
        for j in range(cols):
            if mapa[i][j] == "#":
                continue
            if mapa[i][j] == "S":
                inicio = (i, j)
            if mapa[i][j] == "G":
                meta = (i, j)
            grafo[(i, j)] = {}
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < filas and 0 <= nj < cols and mapa[ni][nj] != "#":
                    grafo[(i, j)][(ni, nj)] = 1
    return grafo, inicio, meta


grafo_mapa, inicio, meta = mapa_a_grafo(mapa)
h_manhattan = {n: abs(n[0] - meta[0]) + abs(n[1] - meta[1]) for n in grafo_mapa}

print(f"{'w':>4} | {'costo':>5} | {'nodos expandidos':>16}")
print("-" * 32)
for w in [1, 1.5, 2, 5]:
    camino, costo, expandidos = a_estrella(grafo_mapa, h_manhattan, inicio, meta, w=w, verbose=False)
    print(f"{w:>4} | {costo:>5} | {expandidos:>16}")

   w | costo | nodos expandidos
--------------------------------
   1 |    29 |               49
 1.5 |    29 |               52
   2 |    29 |               55
   5 |    69 |               80


Vemos que al subir w baja el número de nodos que expandimos. En este laberinto el costo se mantiene, pero no siempre pasa así: con w > 1 ya no hay garantía de que el camino sea el más corto, solo de que no pasa de w veces el óptimo.

Para verlo en el mapa, marcamos con `*` el camino y con `o` los nodos que se expandieron.

In [15]:
def dibujar(mapa, grafo, h, inicio, meta, w):
    # repetimos la búsqueda pero guardando qué nodos se expandieron
    g = {inicio: 0}
    padres = {inicio: None}
    frontera = [(w * h[inicio], inicio)]
    expandidos = set()
    while frontera:
        f, nodo = heapq.heappop(frontera)
        if f > g[nodo] + w * h[nodo]:
            continue
        if nodo == meta:
            break
        expandidos.add(nodo)
        for vecino, peso in grafo[nodo].items():
            if vecino not in g or g[nodo] + peso < g[vecino]:
                g[vecino] = g[nodo] + peso
                padres[vecino] = nodo
                heapq.heappush(frontera, (g[vecino] + w * h[vecino], vecino))

    camino = set(reconstruir_camino(padres, meta))
    for i, fila in enumerate(mapa):
        linea = ""
        for j, c in enumerate(fila):
            if c in "SG#":
                linea += c
            elif (i, j) in camino:
                linea += "*"
            elif (i, j) in expandidos:
                linea += "o"
            else:
                linea += "."
        print(linea)


print("w = 1")
dibujar(mapa, grafo_mapa, h_manhattan, inicio, meta, 1)
print("\nw = 5")
dibujar(mapa, grafo_mapa, h_manhattan, inicio, meta, 5)

w = 1
Sooooooooo#.........
*########o#.######..
*#......#o#......#..
*#.####.#o######.#..
*#.#..#.#oooooo#.#..
*#.#..#.######o#.#..
*#.#..#.......o#.#..
*#.####.########.#..
*#...............#..
*#################..
*******************G

w = 5
S*********#*********
.########*#*######.*
.#....oo#*#******#.*
.#.####o#*######*#.*
.#.#..#o#******#*#.*
.#.#..#o######*#*#.*
.#.#..#********#*#.*
.#o####*########*#.*
.#ooooo**********#.*
.#################.*
...................G


## 10. Minimax

Ahora el problema es un juego de dos jugadores. No podemos saber qué va a hacer el otro, así que suponemos que siempre va a jugar lo mejor para él (lo peor para nosotros).

- **MAX** (nosotros): de sus hijos escoge el valor más alto.
- **MIN** (el oponente): de sus hijos escoge el valor más bajo.

**Qué usa:** recursión sobre el árbol del juego. Las hojas tienen el valor de utilidad (qué tan bueno es ese final para MAX).

**Qué compara:** en cada nodo compara los valores que regresan sus hijos. Si es turno de MAX se queda con el mayor; si es turno de MIN, con el menor. Los turnos se van alternando por nivel.

**Paso a paso:**

1. Si el nodo es una hoja, regresamos su valor.
2. Si no, calculamos minimax de cada hijo, cambiando de turno.
3. Si es MAX regresamos el máximo de esos valores; si es MIN, el mínimo.
4. En la raíz, la jugada que escogemos es la del hijo con el mejor valor.

Los valores suben desde las hojas hasta la raíz. Ojo: minimax revisa **todas** las hojas.

Árbol de ejemplo (el clásico de Russell y Norvig): la raíz es MAX y tiene 3 hijos MIN con hojas [3, 12, 8], [2, 4, 6] y [14, 5, 2].

In [16]:
hojas_revisadas = 0

def minimax(nodo, es_max, nivel=0):
    global hojas_revisadas
    sangria = "   " * nivel

    if isinstance(nodo, (int, float)):   # es una hoja
        hojas_revisadas += 1
        return nodo

    valores = [minimax(hijo, not es_max, nivel + 1) for hijo in nodo]
    if es_max:
        mejor = max(valores)
        print(f"{sangria}MAX compara {valores} y se queda con {mejor}")
    else:
        mejor = min(valores)
        print(f"{sangria}MIN compara {valores} y se queda con {mejor}")
    return mejor


arbol = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]

hojas_revisadas = 0
valor = minimax(arbol, True)
print(f"\nValor del juego: {valor} | hojas revisadas: {hojas_revisadas}")

   MIN compara [3, 12, 8] y se queda con 3
   MIN compara [2, 4, 6] y se queda con 2
   MIN compara [14, 5, 2] y se queda con 2
MAX compara [3, 2, 2] y se queda con 3

Valor del juego: 3 | hojas revisadas: 9


## 11. Poda alfa-beta

Da el mismo resultado que minimax, pero se salta ramas que ya sabemos que no van a cambiar la decisión.

**Qué usa:** la misma recursión que minimax, más dos números que vamos pasando hacia abajo:

- **α (alfa):** lo mejor que MAX ya tiene asegurado en el camino actual. Empieza en -∞ y solo sube.
- **β (beta):** lo mejor que MIN ya tiene asegurado en el camino actual. Empieza en +∞ y solo baja.

**Qué compara:**

- En un nodo MAX: cada vez que un hijo regresa un valor, lo comparamos contra α y si es mayor actualizamos α.
- En un nodo MIN: cada valor de hijo lo comparamos contra β y si es menor actualizamos β.
- Después de cada actualización comparamos **α contra β**. Si α ≥ β, podamos: dejamos de revisar los hijos que faltan de ese nodo.

**Por qué se puede podar:** en el ejemplo, MAX ya sabe que por la primera rama se lleva 3 (α = 3). En la segunda rama, MIN ve una hoja con 2, así que MIN va a regresar 2 o menos (β = 2). Como MAX ya tiene 3, nunca va a escoger esa rama, así que no importa cuánto valgan las otras hojas (4 y 6). No las revisamos.

In [17]:
import math

def alfa_beta(nodo, es_max, alfa=-math.inf, beta=math.inf, nivel=0):
    global hojas_revisadas
    sangria = "   " * nivel

    if isinstance(nodo, (int, float)):
        hojas_revisadas += 1
        print(f"{sangria}hoja {nodo}")
        return nodo

    tipo = "MAX" if es_max else "MIN"
    print(f"{sangria}{tipo} entra con alfa = {alfa}, beta = {beta}")
    mejor = -math.inf if es_max else math.inf

    for i, hijo in enumerate(nodo):
        valor = alfa_beta(hijo, not es_max, alfa, beta, nivel + 1)

        if es_max:
            mejor = max(mejor, valor)
            alfa = max(alfa, mejor)
        else:
            mejor = min(mejor, valor)
            beta = min(beta, mejor)

        if alfa >= beta:
            print(f"{sangria}{tipo}: alfa ({alfa}) >= beta ({beta}) -> podamos {nodo[i + 1:]}")
            break

    print(f"{sangria}{tipo} regresa {mejor}")
    return mejor


hojas_revisadas = 0
valor = alfa_beta(arbol, True)
print(f"\nValor del juego: {valor} | hojas revisadas: {hojas_revisadas} de 9")

MAX entra con alfa = -inf, beta = inf
   MIN entra con alfa = -inf, beta = inf
      hoja 3
      hoja 12
      hoja 8
   MIN regresa 3
   MIN entra con alfa = 3, beta = inf
      hoja 2
   MIN: alfa (3) >= beta (2) -> podamos [4, 6]
   MIN regresa 2
   MIN entra con alfa = 3, beta = inf
      hoja 14
      hoja 5
      hoja 2
   MIN: alfa (3) >= beta (2) -> podamos []
   MIN regresa 2
MAX regresa 3

Valor del juego: 3 | hojas revisadas: 7 de 9


Mismo valor (3) que minimax, pero revisando menos hojas. Entre mejor ordenados estén los hijos (los mejores primero), más se poda.

## Búsqueda local: problema de las 8 reinas

Para los últimos algoritmos ya no buscamos un camino, sino un **estado** que sea bueno. No guardamos árbol ni frontera, solo el estado actual (o una población de estados).

Usamos las 8 reinas. Representamos el tablero como una lista de 8 números: `estado[col] = fila` donde está la reina de esa columna. Así ya hay una reina por columna y solo pueden chocar por fila o por diagonal.

La función que medimos es el número de **pares de reinas que se atacan**. Queremos llegar a 0.

- Mismo renglón: `estado[i] == estado[j]`
- Misma diagonal: `abs(estado[i] - estado[j]) == abs(i - j)`

In [18]:
import random

def conflictos(estado):
    n = len(estado)
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            if estado[i] == estado[j] or abs(estado[i] - estado[j]) == j - i:
                total += 1
    return total


def imprimir_tablero(estado):
    n = len(estado)
    for fila in range(n):
        print(" ".join("R" if estado[col] == fila else "." for col in range(n)))

## 12. Hill Climbing (ascenso de colinas)

Es como estar en una montaña con neblina: solo vemos lo que está alrededor de nuestros pies y damos el paso hacia donde mejora más.

**Qué usa:** solo el estado actual.

**Vecinos:** todos los tableros que salen de mover **una** reina a otra fila dentro de su misma columna (8 columnas × 7 filas = 56 vecinos).

**Qué compara:**

- Calcula los conflictos de los 56 vecinos y escoge el que tenga menos.
- Compara los conflictos de ese mejor vecino **contra los conflictos del estado actual**. Si el vecino es mejor, nos movemos. Si no es mejor, nos detenemos.

**Paso a paso:**

1. Empezamos con un tablero al azar.
2. Generamos todos los vecinos y calculamos sus conflictos.
3. Tomamos el vecino con menos conflictos.
4. Si tiene menos conflictos que el actual, nos movemos a él y regresamos al paso 2.
5. Si no, paramos. Puede ser la solución (0 conflictos) o un **mínimo local** (ningún vecino mejora pero todavía hay ataques).

**Problema:** se atora muy seguido en mínimos locales. La forma más simple de arreglarlo es **reiniciar** desde otro tablero al azar hasta encontrar la solución.

In [19]:
def hill_climbing(estado, verbose=True):
    actual = list(estado)
    paso = 0

    while True:
        c_actual = conflictos(actual)
        mejor_vecino, c_mejor = None, c_actual

        for col in range(8):
            for fila in range(8):
                if fila == actual[col]:
                    continue
                vecino = list(actual)
                vecino[col] = fila
                c = conflictos(vecino)
                if c < c_mejor:
                    mejor_vecino, c_mejor = vecino, c

        if mejor_vecino is None:
            if verbose:
                print(f"Ningún vecino mejora los {c_actual} conflictos -> paramos")
            return actual

        paso += 1
        if verbose:
            print(f"Paso {paso}: {c_actual} conflictos -> nos movemos a un vecino con {c_mejor}")
        actual = mejor_vecino


random.seed(1)
inicio = [random.randint(0, 7) for _ in range(8)]
print("Inicio:", inicio, "conflictos:", conflictos(inicio))
final = hill_climbing(inicio)
print("Final:", final, "conflictos:", conflictos(final))

Inicio: [2, 1, 4, 1, 7, 7, 7, 6] conflictos: 9
Paso 1: 9 conflictos -> nos movemos a un vecino con 5
Paso 2: 5 conflictos -> nos movemos a un vecino con 3
Paso 3: 3 conflictos -> nos movemos a un vecino con 2
Paso 4: 2 conflictos -> nos movemos a un vecino con 1
Ningún vecino mejora los 1 conflictos -> paramos
Final: [5, 2, 4, 1, 7, 0, 3, 6] conflictos: 1


### Con reinicios aleatorios

Si se queda atorado, empezamos otra vez desde un tablero nuevo.

In [20]:
def hill_climbing_reinicios(max_intentos=100):
    for intento in range(1, max_intentos + 1):
        inicio = [random.randint(0, 7) for _ in range(8)]
        final = hill_climbing(inicio, verbose=False)
        c = conflictos(final)
        print(f"Intento {intento}: terminó con {c} conflictos")
        if c == 0:
            return final
    return None


random.seed(1)
solucion = hill_climbing_reinicios()
print()
imprimir_tablero(solucion)

Intento 1: terminó con 1 conflictos
Intento 2: terminó con 1 conflictos
Intento 3: terminó con 1 conflictos
Intento 4: terminó con 0 conflictos

. . . R . . . .
. R . . . . . .
. . . . R . . .
. . . . . . . R
. . . . . R . .
R . . . . . . .
. . R . . . . .
. . . . . . R .


## 13. Recocido simulado (Simulated Annealing)

Sale de la metalurgia: el metal caliente se mueve mucho y al enfriarse se va acomodando. Aquí la "temperatura" T controla qué tanto nos dejamos hacer malas jugadas.

**Qué usa:** el estado actual y la temperatura T, que va bajando en cada paso (T = T · enfriamiento).

**Qué compara:**

1. Escogemos **un** vecino al azar (no todos como en Hill Climbing).
2. Calculamos `delta = conflictos(vecino) - conflictos(actual)`.
3. Si delta < 0 (el vecino es mejor), nos movemos siempre.
4. Si delta ≥ 0 (es igual o peor), sacamos un número al azar entre 0 y 1 y lo comparamos contra la probabilidad `e^(-delta / T)`. Si el número es menor, aceptamos el paso malo de todos modos.

**Qué pasa con T:**

- Al principio T es grande, entonces e^(-delta / T) es casi 1 y aceptamos casi cualquier movimiento. Eso nos deja salir de mínimos locales.
- Conforme T baja, esa probabilidad se va a 0 y ya solo aceptamos mejoras, igual que Hill Climbing.

Paramos cuando llegamos a 0 conflictos o cuando T ya es muy chica.

In [21]:
def recocido_simulado(estado, T=5.0, enfriamiento=0.995, T_min=0.001):
    actual = list(estado)
    c_actual = conflictos(actual)
    paso = 0

    while T > T_min and c_actual > 0:
        vecino = list(actual)
        col = random.randint(0, 7)
        vecino[col] = random.choice([f for f in range(8) if f != actual[col]])
        c_vecino = conflictos(vecino)
        delta = c_vecino - c_actual

        if delta < 0:
            actual, c_actual = vecino, c_vecino
        else:
            prob = math.exp(-delta / T)
            if random.random() < prob:
                actual, c_actual = vecino, c_vecino

        if paso % 200 == 0:
            print(f"paso {paso:4d} | T = {T:.3f} | conflictos = {c_actual}")
        T *= enfriamiento
        paso += 1

    print(f"Terminamos en el paso {paso} con T = {T:.3f} y {c_actual} conflictos")
    return actual


random.seed(1)
inicio = [random.randint(0, 7) for _ in range(8)]
final = recocido_simulado(inicio)
print()
imprimir_tablero(final)

paso    0 | T = 5.000 | conflictos = 8
paso  200 | T = 1.835 | conflictos = 8
paso  400 | T = 0.673 | conflictos = 4
paso  600 | T = 0.247 | conflictos = 1
paso  800 | T = 0.091 | conflictos = 1
paso 1000 | T = 0.033 | conflictos = 1
Terminamos en el paso 1181 con T = 0.013 y 0 conflictos

. . R . . . . .
. . . . R . . .
. . . . . . . R
. . . R . . . .
R . . . . . . .
. . . . . . R .
. R . . . . . .
. . . . . R . .


Para ver cómo cambia la probabilidad de aceptar un paso que empeora 1 conflicto (delta = 1) según la temperatura:

In [22]:
for T in [5, 2, 1, 0.5, 0.1]:
    print(f"T = {T:>4}: probabilidad de aceptar = {math.exp(-1 / T):.3f}")

T =    5: probabilidad de aceptar = 0.819
T =    2: probabilidad de aceptar = 0.607
T =    1: probabilidad de aceptar = 0.368
T =  0.5: probabilidad de aceptar = 0.135
T =  0.1: probabilidad de aceptar = 0.000
